[VideoQA](https://www.kaggle.com/benchmarks/deepmind/video-qa/leaderboard) is a benchmark from Google DeepMind. It is a long-context multimodal benchmark designed to measure long-form video understanding.

Please check out this [Research Paper](https://arxiv.org/abs/2403.05530) for further details!

**This notebook demonstrates how to evaluate Large Vision-Language Models (VLMs) on the VideoQA benchmark.** We will walk through preparing the 1-hour video dataset, handling frame extraction for different API constraints, and running a robust automated evaluation to measure reasoning accuracy.

### Install Dependencies

In [ ]:
!pip install -q opencv-python-headless
import os

In [ ]:
import re
import base64
import random
import tempfile
import subprocess
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import os

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images, videos

In [ ]:
public_df = pd.read_csv("/kaggle/input/datasets/organizations/deepmind/videoqa/1hvideoqa.csv")
private_df = pd.read_csv("/kaggle/input/datasets/organizations/deepmind/videoqa-with-answers/1hvideoqa_with_answers.csv")

df = public_df.merge(private_df, on=["Video URL", "Question"])

eval_df = df[["Video URL", "Prompt_x", "Label", "Capability_x"]].rename(columns={
    "Video URL": "video_url",
    "Prompt_x": "prompt",
    "Label": "label",
    "Capability_x": "capability",
})


### Model Provider Configurations

In [ ]:
SEED = 0

# Per-provider max frames
PROVIDER_MAX_FRAMES = {
    "openai": 500,
    "anthropic": 600,
    "gemini": None, # 2896
    "default": 500,
}

# Per-provider image budget config
PROVIDER_IMAGE_CONFIG = {
    "openai": {"max_total_mb": 31.0, "max_dim": 512, "per_image": False, "extra_api_params": {"detail": "low"}},
    "anthropic": {"max_total_mb": 20.0, "max_dim": 512, "per_image": False},
    "gemini": {"max_total_mb": 100000.0, "max_dim": None, "per_image": False, "extra_api_params": {"media_resolution": {"level": "MEDIA_RESOLUTION_LOW"}}},
    "default": {"max_total_mb": 30.0, "max_dim": 512, "per_image": False},
}

def get_max_frames(llm) -> int:
    name = llm.name.lower()
    if "gemini" in name:
        return PROVIDER_MAX_FRAMES["gemini"]
    elif "gpt" in name or "openai" in name or "o3" in name or "o4" in name:
        return PROVIDER_MAX_FRAMES["openai"]
    elif "claude" in name or "anthropic" in name:
        return PROVIDER_MAX_FRAMES["anthropic"]
    else:
        return PROVIDER_MAX_FRAMES["default"]
def get_image_config(llm) -> dict:
    name = llm.name.lower()
    if "gemini" in name:
        return PROVIDER_IMAGE_CONFIG["gemini"]
    elif "gpt" in name or "openai" in name or "o3" in name or "o4" in name:
        return PROVIDER_IMAGE_CONFIG["openai"]
    elif "claude" in name or "anthropic" in name:
        return PROVIDER_IMAGE_CONFIG["anthropic"]
    else:
        return PROVIDER_IMAGE_CONFIG["default"]

def supports_video(llm) -> bool:
    return "gemini" in llm.name.lower()

### Video Processing & Frame Extraction

Since not all models support native video file ingestion, we need a way to pass the video visual data to them. 

This section defines utilities to:
1. **Locate** the pre-downloaded video files on Kaggle.
2. **Extract** frames evenly across the video using OpenCV (`cv2`). We use a linearly spaced sampling method to ensure the extracted frames represent the entire video without exceeding the provider's frame limit.
3. **Encode** the extracted frames into base64 JPEG format, scaling them down if necessary to meet dimension budgets.

In [ ]:
VIDEO_DIR = "/kaggle/input/datasets/organizations/deepmind/videoqa-videos"

def get_video_path(video_url: str) -> str:
    """Find pre-downloaded video file by YouTube ID."""
    vid_id = video_url.split("watch?v=")[-1]
    for ext in ["mp4", "webm", "mkv"]:
        path = os.path.join(VIDEO_DIR, f"{vid_id}.{ext}")
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Video not found: {vid_id}")


In [ ]:
def encode_frame(frame_bgr, max_dim: int | None = None, quality: int = 80) -> str:
    """Encode frame as JPEG base64. Optionally cap max dimension."""
    h, w = frame_bgr.shape[:2]

    # Only resize if max_dim is set (e.g. OpenAI)
    if max_dim and max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        frame_bgr = cv2.resize(frame_bgr, (int(w * scale), int(h * scale)),
                               interpolation=cv2.INTER_AREA)

    _, buffer = cv2.imencode(".jpg", frame_bgr, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return base64.b64encode(buffer.tobytes()).decode()


def extract_frames(video_path: str, max_frames: int, max_dim: int | None = None, extra_api_params: dict | None = None) -> list[tuple[str, object]]: 
    """Extract frames using FPS_LINSPACE_UNIQUE."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # 1fps indices
    stride = fps / 1.0
    num_1fps = max(int(total_frames / stride), 1)
    fps_indices = np.floor(np.arange(0, int(num_1fps * stride), stride)).astype(int)

    # Linspace subsample if needed
    if len(fps_indices) > max_frames:
        subsample = np.unique(
            np.floor(np.linspace(0, len(fps_indices) - 1, max_frames)).astype(int)
        )
        fps_indices = fps_indices[subsample]

    extracted = []
    for i, target_idx in enumerate(fps_indices):
        if i % 500 == 0:                                                                                                                                                   
          print(f"[DEBUG] Extracting frame {i}/{len(fps_indices)}")
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(target_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        seconds = target_idx / fps
        minutes, secs = divmod(int(seconds), 60)
        timestamp = f"{minutes:02d}:{secs:02d}"

        b64 = encode_frame(frame, max_dim=max_dim)
        image = images.from_base64(b64, format="jpeg", extra_api_params=extra_api_params or {})
        extracted.append((timestamp, image))

    cap.release()
    return extracted


### Response Parsing

In [ ]:
# Thought boundary patterns 
THOUGHT_PATTERNS = [
    re.compile(r"<ctrl94>thought.*?<ctrl95>", flags=re.DOTALL),
    re.compile(r"<ctrl3347>.*?<ctrl3348>", flags=re.DOTALL),
    re.compile(r"<thought>.*?</thought>", flags=re.DOTALL),
    re.compile(r"<think>.*?</think>", flags=re.DOTALL),
]

LABELS = ["(A)", "(B)", "(C)", "(D)", "(E)"]
LABELS_LOWER = [l.lower() for l in LABELS]

# Regex: (a)-(e) with parens | $\boxed{a-e}$ | bare a-e word boundary
ANSWER_PATTERN = re.compile(
    r"\(([abcde])\)|\$\\boxed\{([abcde])\}|\b([abcde])\b"
)


def remove_thoughts(text: str) -> str:
    """Remove thinking/thought tokens from model output."""
    for pattern in THOUGHT_PATTERNS:
        text = pattern.sub("", text)
    for start_marker in ["<ctrl94>thought", "<ctrl3347>", "<thought>", "<think>"]:
        if start_marker in text:
            text = text.split(start_marker)[0]
    return text


def extract_answer(response: str) -> str:
    """Extract answer from response.

    Returns label in "(X)" format (lowercase), e.g. "(a)", "(b)".
    If no valid answer found, returns a random label.
    """
    cleaned = remove_thoughts(response)
    cleaned = cleaned.lower()
    cleaned = cleaned.replace("*", "")

    # Strip content after <ctrl100>
    ctrl100_idx = cleaned.find("<ctrl100>")
    if ctrl100_idx > 0:
        cleaned = cleaned[:ctrl100_idx]

    # Prioritize text after "final answer" if present
    if "final answer" in cleaned:
        cleaned = cleaned.split("final answer")[-1]

    match = ANSWER_PATTERN.search(cleaned)
    if match:
        letter = match.group(1) or match.group(2) or match.group(3)
        model_response = f"({letter})"
        if model_response in LABELS_LOWER:
            return model_response

    # Invalid prediction -> random guess
    return random.choice(LABELS_LOWER)  # noqa: S311 - seeded at task start


### Row-Level Evaluation (`video_qa`)
For every row in our dataset, this task:
* Checks the model provider to determine if it should send the native video or a series of extracted base64 frames.
* Prompts the LLM with the visual data and the question.
* Compares the parsed prediction against the expected label.
* Aggressively clears memory (`gc.collect()`) to prevent the notebook from running out of RAM on large 1-hour videos.

In [ ]:
@kbench.task(name="video_qa", store_task=False, store_run=False)
def video_qa(llm, video_url, prompt, label, capability) -> dict:
    random.seed(SEED)
    try:
        max_frames = get_max_frames(llm)
        # temp = 0.0 if "gemini" in llm.name.lower() else 0.5

        if max_frames is None:
            response = llm.prompt(prompt, video=videos.from_url(video_url), temperature=1.0, reasoning="high")
        else:
            img_cfg = get_image_config(llm)
            video_path = get_video_path(video_url)
            # print(f"[DEBUG] Loaded from {video_path}")
            frames = extract_frames(
                video_path,
                max_frames=max_frames,
                max_dim=img_cfg["max_dim"],                                                                                                                                                                                
                extra_api_params=img_cfg.get("extra_api_params", {}), 
            )
            # print(f"[DEBUG] Sending {len(frames)} frames...")
            for timestamp, frame in frames:
                kbench.user.send(timestamp)
                kbench.user.send(frame)
            response = llm.prompt(prompt, temperature=1.0, reasoning="high")
            # Free image memory from chat
            chat = kbench.chats.get_current_chat()
            chat.history.clear()                                                                                                                               
            del frames
            import gc; gc.collect() 
        # print(f"[DEBUG] response: {response}")
        predicted = extract_answer(response)
        expected = label.strip().lower()
        is_correct = predicted == expected
        print(f"[DEBUG] predicted={predicted}, expected={expected}, correct={is_correct}")

        return {
            "predicted_answer": predicted,
            "is_correct": is_correct,
            "capability": capability,
            "error": None,
        }
    except Exception as e:                                                                                                                                                                                                           
        import traceback                                                                                                                                                                                                                 
        traceback.print_exc()  
        print(f"[DEBUG] ERROR: {type(e).__name__}: {e}")
        return {
            "predicted_answer": None,
            "is_correct": None,
            "capability": capability,
            "error": str(e),
        }


### Aggregate Evaluation (`video_qa_benchmark`)
This task runs the row-level evaluation across the entire dataset, handles error reporting for failed API calls, and calculates the final accuracy, segmenting performance by capability (Recall vs. Reasoning).

In [ ]:
@kbench.task(name="video_qa_benchmark")
def video_qa_benchmark(llm, evaluation_data, n_jobs=1) -> float:
    llm.support_temperature = True
    with kbench.client.enable_cache():
        runs = video_qa.evaluate(
            llm=[llm],
            evaluation_data=evaluation_data,
            n_jobs=n_jobs,
            remove_run_files=False,
        )
    
    eval_df = runs.as_dataframe()
    for run in runs.runs:
      if run.chat:                                                                                                                         
          run.chat.history.clear()                                                                                                         
      run.subruns.clear()                                                                                                                  
    import gc; gc.collect()  
    # Filter out failed rows
    errors = eval_df.result.str.get("error")
    n_errors = errors.notna().sum()
    if n_errors > 0:
        print(f"Warning: {n_errors}/{len(eval_df)} rows failed. Errors:")
        for err in errors.dropna().unique():
            print(f"  - {err}")

    valid = errors.isna()
    is_correct = eval_df.result.str.get("is_correct")[valid].astype(bool)
    capability = eval_df.result.str.get("capability")[valid]

    recall_mask = capability == "Recall"
    reasoning_mask = ~recall_mask

    n = len(is_correct)
    accuracy = float(is_correct.mean()) if n > 0 else 0.0
    se = float(is_correct.std() / (n ** 0.5)) if n > 1 else 0.0

    recall_acc = float(is_correct[recall_mask].mean()) if recall_mask.any() else 0.0
    reasoning_acc = float(is_correct[reasoning_mask].mean()) if reasoning_mask.any() else 0.0

    return accuracy


In [ ]:
# llm = kbench.kaggle.load_model("anthropic/claude-sonnet-4-6@default")  # 
video_qa_benchmark.run(kbench.llm, evaluation_data=eval_df)

In [ ]:
%choose video_qa_benchmark